## Insider Threat Radar – Behavior Analytics for Employee Risk Prediction

### Time-Series GRU

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

#### Load Data

In [2]:
email_df = pd.read_csv("../Dataset/email_clean.csv")
email_df['date'] = pd.to_datetime(email_df['date'])
email_df['day'] = email_df['date'].dt.date

daily = email_df.groupby(['user','day']).agg({
    'size':'mean',
    'attachments':'sum'
}).reset_index()

#### Scaling

In [3]:
scaler = MinMaxScaler()
daily[['size','attachments']] = scaler.fit_transform(
    daily[['size','attachments']]
)

#### Sequence Creation

In [4]:
sequence_length = 10
X, y = [], []

for user in daily['user'].unique():
    user_data = daily[daily['user']==user][['size','attachments']].values
    for i in range(len(user_data)-sequence_length):
        X.append(user_data[i:i+sequence_length])
        y.append(user_data[i+sequence_length][0])

X = np.array(X)
y = np.array(y)

#### Train GRU

In [5]:
gru = Sequential([
    GRU(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.2),
    GRU(32),
    Dense(1)
])

gru.compile(optimizer='adam', loss='mse')
gru.fit(X, y, epochs=10, batch_size=64, validation_split=0.2)



Epoch 1/10

3963/3963 [==============================] - 42s 10ms/step - loss: 0.0026 - val_loss: 0.0023
Epoch 2/10
3963/3963 [==============================] - 40s 10ms/step - loss: 0.0025 - val_loss: 0.0022
Epoch 3/10
3963/3963 [==============================] - 39s 10ms/step - loss: 0.0025 - val_loss: 0.0022
Epoch 4/10
3963/3963 [==============================] - 40s 10ms/step - loss: 0.0025 - val_loss: 0.0022
Epoch 5/10
3963/3963 [==============================] - 39s 10ms/step - loss: 0.0025 - val_loss: 0.0022
Epoch 6/10
3963/3963 [==============================] - 40s 10ms/step - loss: 0.0025 - val_loss: 0.0022
Epoch 7/10
3963/3963 [==============================] - 42s 11ms/step - loss: 0.0025 - val_loss: 0.0023
Epoch 8/10
3963/3963 [==============================] - 39s 10ms/step - loss: 0.0025 - val_loss: 0.0023
Epoch 9/10
3963/3963 [==============================] - 41s 10ms/step - loss: 0.0025 - val_loss: 0.0023
Epoch 10/10
3963/3963 [==============================] - 40s 

#### Prediction

In [6]:
y_pred = gru.predict(X)

print("GRU MSE:", mean_squared_error(y, y_pred))

9906/9906 [==============================] - 31s 3ms/step
GRU MSE: 0.0024836857842415344


#### Save model

In [7]:
gru.save("../Model/gru_model.h5")
print("GRU model saved ✅")

GRU model saved ✅


C:\Users\viqua\anaconda3\envs\pilot\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
